# MedAI — Kaggle P100 QLoRA training notebook

**Setup:** Kaggle → Settings → Accelerator: **GPU P100** (16 GB), Internet **ON**.

Add Kaggle Secrets:
- `HF_TOKEN` — your HuggingFace write token (for checkpoint upload)
- `WANDB_API_KEY` — optional, for tracking

9-hour session limit: this notebook checkpoints every 500 steps to HF Hub. If killed, run `[Resume]` cell.

## 1. Install

In [ ]:
%%capture
!pip install -q --upgrade pip
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes
!pip install -q datasets pyyaml huggingface_hub

## 2. Clone repo + auth

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')

from huggingface_hub import login
login(token=os.environ['HF_TOKEN'])

!git clone https://github.com/vh0202/MedAI.git /kaggle/working/MedAI
%cd /kaggle/working/MedAI

## 3. Tokenizer sanity check (Phase 0 gate)

In [ ]:
!python eval/tokenizer_test.py --model unsloth/Qwen2.5-7B-Instruct-bnb-4bit

## 4. Prepare bilingual data

In [ ]:
# Phase 1: ~30k EN + 20k VI (×2 upsample) ≈ 70k samples
!python data/prepare_bilingual_mix.py --phase 1 --out data/phase1_mix.jsonl

## 5. (Optional) Translate EN→VI to enrich VN data

Skip if you already have `data/translation_qa/en2vi.jsonl` from a previous run.

In [ ]:
# Translate ~25k EN samples to VI (~30-60 min on P100)
# !python data/translate_nllb.py --input data/english/en_subset.jsonl \
#     --output data/translation_qa/en2vi.jsonl --limit 25000

## 6. Train (initial run)

In [ ]:
!python scripts/train_sft.py \
    --config configs/qwen25_7b_qlora.yaml \
    --data data/phase1_mix.jsonl

## 6b. [Resume] — if Kaggle killed the session

Spin up a NEW Kaggle session, run cells 1–5 above, then this cell instead of cell 6.

In [ ]:
!python scripts/resume_training.py \
    --config configs/qwen25_7b_qlora.yaml \
    --data data/phase1_mix.jsonl \
    --hub_repo vh0202/medai-qwen25-7b-phase1-ckpt

## 7. Quick eval after training

In [ ]:
!python eval/medqa_eval.py --adapter outputs/qwen25-7b-medai-phase1 --limit 200
!python eval/vn_medical_benchmark/run_eval.py --adapter outputs/qwen25-7b-medai-phase1